In [0]:
# ============================================================
# NOTEBOOK 03: CREATE SILVER DQ RULES CONFIG
# Purpose:
#   1. Create the shared Silver DQ metadata Delta table.
#   2. Read the active schema for the selected dataset.
#   3. Validate proposed DQ columns against that schema.
#   4. Insert new DQ rules and update existing rules.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    MapType,
    TimestampType,
    IntegerType,
    BooleanType
)
from delta.tables import DeltaTable

In [0]:
# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
)

SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"
DQ_CONFIG_PATH = BASE_PATH + "metadata/silver_dq_rules_config/"

dbutils.widgets.text("dataset_name", "google_ads_campaigns")
dataset_name = dbutils.widgets.get("dataset_name").strip()

if not dataset_name:
    raise ValueError("dataset_name cannot be empty")

print(f"Dataset selected: {dataset_name}")

Dataset selected: google_ads_campaigns


In [0]:
# ------------------------------------------------------------
# 2. READ ACTIVE SCHEMA CONFIGURATION
# ------------------------------------------------------------

schema_config_df = (
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
)

active_schema_rows = (
    schema_config_df
    .filter(
        (F.col("dataset_name") == dataset_name) &
        (F.col("is_active") == True)
    )
    .collect()
)

if len(active_schema_rows) != 1:
    raise ValueError(
        f"Expected exactly one active schema configuration "
        f"for {dataset_name}, found {len(active_schema_rows)}"
    )

schema_config = active_schema_rows[0]

schema_ddl = schema_config["schema_ddl"]
registered_schema = StructType.fromDDL(schema_ddl)

registered_columns = list(registered_schema.fieldNames())

print("Active schema found.")
print(f"Registered columns: {len(registered_columns)}")
print(registered_columns)

Active schema found.
Registered columns: 18
['customer_id', 'campaign_id', 'campaign_resource_name', 'campaign_name', 'campaign_status', 'advertising_channel_type', 'advertising_channel_sub_type', 'campaign_budget_resource_name', 'bidding_strategy_type', 'start_date', 'end_date', 'campaign_primary_status', 'serving_status', 'target_country', 'business_brand', 'product_category', 'business_objective', 'source_extract_timestamp']


In [0]:
# ------------------------------------------------------------
# 3. DEFINE THE DQ METADATA TABLE SCHEMA
# ------------------------------------------------------------

dq_table_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("rule_id", StringType(), False),
    StructField("column_name", StringType(), True),
    StructField("rule_type", StringType(), False),
    StructField(
        "rule_parameters",
        MapType(StringType(), StringType()),
        True
    ),
    StructField("severity", StringType(), False),
    StructField("failure_action", StringType(), False),
    StructField("execution_order", IntegerType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("rule_status", StringType(), False),
    StructField("description", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

In [0]:
# ------------------------------------------------------------
# 4. CREATE DELTA TABLE IF IT DOES NOT EXIST
# ------------------------------------------------------------

if not DeltaTable.isDeltaTable(spark, DQ_CONFIG_PATH):

    empty_dq_df = spark.createDataFrame(
        [],
        dq_table_schema
    )

    (
        empty_dq_df.write
        .format("delta")
        .mode("overwrite")
        .save(DQ_CONFIG_PATH)
    )

    print("Created Silver DQ rules configuration table.")

else:
    print("Silver DQ rules configuration table already exists.")

Silver DQ rules configuration table already exists.


In [0]:
# ------------------------------------------------------------
# 5. DEFINE PROPOSED DATA QUALITY RULES
# ------------------------------------------------------------

def make_rule(
    rule_id,
    column_name,
    rule_type,
    execution_order,
    description,
    rule_parameters=None,
    severity="HIGH",
    failure_action="QUARANTINE"
):
    return {
        "dataset_name": dataset_name,
        "rule_id": rule_id,
        "column_name": column_name,
        "rule_type": rule_type,
        "rule_parameters": rule_parameters or {},
        "severity": severity,
        "failure_action": failure_action,
        "execution_order": execution_order,
        "is_active": True,
        "rule_status": "APPROVED",
        "description": description
    }


proposed_rules = [

    # A. REQUIRED FIELD CHECKS

    make_rule(
        "CHK_CUSTOMER_ID_NOT_NULL",
        "customer_id",
        "NOT_NULL",
        1,
        "Customer ID must not be null."
    ),

    make_rule(
        "CHK_CAMPAIGN_ID_NOT_NULL",
        "campaign_id",
        "NOT_NULL",
        2,
        "Campaign ID must not be null."
    ),

    make_rule(
        "CHK_CAMPAIGN_RESOURCE_NOT_NULL",
        "campaign_resource_name",
        "NOT_NULL",
        3,
        "Campaign resource name must not be null."
    ),

    make_rule(
        "CHK_SOURCE_EXTRACT_TS_NOT_NULL",
        "source_extract_timestamp",
        "NOT_NULL",
        4,
        "Source extraction timestamp must not be null."
    ),


    # B. UNIQUENESS CHECK

    make_rule(
        "CHK_CAMPAIGN_RESOURCE_UNIQUE",
        "campaign_resource_name",
        "UNIQUE",
        5,
        "Campaign resource name must be unique."
    ),


    # C. ACCEPTED VALUES CHECKS

    make_rule(
        "CHK_CAMPAIGN_STATUS_VALUES",
        "campaign_status",
        "ACCEPTED_VALUES",
        6,
        "Campaign status must belong to the approved values.",
        {
            "accepted_values": "ENABLED,PAUSED,REMOVED"
        },
        severity="MEDIUM"
    ),

    make_rule(
        "CHK_ADVERTISING_CHANNEL_VALUES",
        "advertising_channel_type",
        "ACCEPTED_VALUES",
        7,
        "Advertising channel type must be an approved value.",
        {
            "accepted_values": "SEARCH,DISPLAY,VIDEO,SHOPPING,PERFORMANCE_MAX"
        },
        severity="MEDIUM"
    ),

    make_rule(
        "CHK_CAMPAIGN_PRIMARY_STATUS_VALUES",
        "campaign_primary_status",
        "ACCEPTED_VALUES",
        8,
        "Campaign primary status must be an approved value.",
        {
            "accepted_values": "PAUSED,LIMITED,ELIGIBLE"
        },
        severity="MEDIUM"
    ),

    make_rule(
        "CHK_SERVING_STATUS_VALUES",
        "serving_status",
        "ACCEPTED_VALUES",
        9,
        "Serving status must be an approved value.",
        {
            "accepted_values": "NOT_SERVING,UNKNOWN,SERVING"
        },
        severity="MEDIUM"
    ),


    # D. FORMAT CHECK

    make_rule(
        "CHK_TARGET_COUNTRY_FORMAT",
        "target_country",
        "REGEX",
        10,
        "Target country must follow a two-letter uppercase code format.",
        {
            "pattern": "^[A-Z]{2}$"
        },
        severity="MEDIUM"
    ),


    # E. OPTIONAL FIELD FORMAT CHECK

    make_rule(
        "CHK_CAMPAIGN_NAME_NOT_NULL",
        "campaign_name",
        "NOT_NULL",
        11,
        "Campaign name must not be null."
    )

]

In [0]:

# ------------------------------------------------------------
# 6. VALIDATE RULE DEFINITIONS
# ------------------------------------------------------------

SUPPORTED_RULE_TYPES = {
    "NOT_NULL",
    "UNIQUE",
    "ACCEPTED_VALUES",
    "REGEX",
    "MIN_VALUE",
    "MAX_VALUE",
    "RANGE",
    "REFERENTIAL_INTEGRITY"
}

SUPPORTED_SEVERITIES = {
    "CRITICAL",
    "HIGH",
    "MEDIUM",
    "LOW"
}

SUPPORTED_FAILURE_ACTIONS = {
    "QUARANTINE",
    "FAIL_DATASET",
    "WARN"
}

# Rule types that require specific parameters.
REQUIRED_PARAMETERS = {
    "ACCEPTED_VALUES": {"accepted_values"},
    "REGEX": {"pattern"},
    "MIN_VALUE": {"min_value"},
    "MAX_VALUE": {"max_value"},
    "RANGE": {"min_value", "max_value"},
    "REFERENTIAL_INTEGRITY": {
        "reference_dataset",
        "reference_column"
    }
}

seen_rule_ids = set()

for rule in proposed_rules:

    # 1. Validate rule ID
    rule_id = rule["rule_id"]

    if not rule_id:
        raise ValueError("Rule ID cannot be empty.")

    if rule_id in seen_rule_ids:
        raise ValueError(f"Duplicate rule ID: {rule_id}")

    seen_rule_ids.add(rule_id)

    # 2. Validate rule type
    if rule["rule_type"] not in SUPPORTED_RULE_TYPES:
        raise ValueError(
            f"Unsupported rule type: {rule['rule_type']}"
        )

    # 3. Validate severity
    if rule["severity"] not in SUPPORTED_SEVERITIES:
        raise ValueError(
            f"Unsupported severity: {rule['severity']}"
        )

    # 4. Validate failure action
    if rule["failure_action"] not in SUPPORTED_FAILURE_ACTIONS:
        raise ValueError(
            f"Unsupported failure action: {rule['failure_action']}"
        )

    # 5. Validate column existence
    column_name = rule["column_name"]

    if not column_name:
        raise ValueError(
            f"Rule {rule_id} must specify a column_name."
        )

    if column_name not in registered_columns:
        raise ValueError(
            f"Column '{column_name}' does not exist in the "
            f"registered schema for {dataset_name}."
        )

    # 6. Validate rule parameters
    parameters = rule["rule_parameters"] or {}

    required = REQUIRED_PARAMETERS.get(
        rule["rule_type"], set()
    )

    missing_parameters = required - set(parameters.keys())

    if missing_parameters:
        raise ValueError(
            f"Rule {rule_id} is missing parameters: "
            f"{missing_parameters}"
        )

    # 7. Validate execution order
    if rule["execution_order"] < 1:
        raise ValueError(
            f"Invalid execution order for {rule_id}."
        )

print(
    f"All {len(proposed_rules)} proposed DQ rules "
    "passed metadata validation."
)


All 11 proposed DQ rules passed metadata validation.


In [0]:
# ------------------------------------------------------------
# 7. CREATE SOURCE DATAFRAME
# ------------------------------------------------------------

rule_rows = [
    (
        r["dataset_name"],
        r["rule_id"],
        r["column_name"],
        r["rule_type"],
        r["rule_parameters"],
        r["severity"],
        r["failure_action"],
        r["execution_order"],
        r["is_active"],
        r["rule_status"],
        r["description"],
        None,
        None
    )
    for r in proposed_rules
]

source_dq_df = spark.createDataFrame(
    rule_rows,
    dq_table_schema
)

source_dq_df = (
    source_dq_df
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
# ------------------------------------------------------------
# 8. INSERT / UPDATE RULES USING DELTA MERGE
# ------------------------------------------------------------

target = DeltaTable.forPath(
    spark,
    DQ_CONFIG_PATH
)

(
    target.alias("target")
    .merge(
        source_dq_df.alias("source"),
        """
        target.dataset_name = source.dataset_name
        AND target.rule_id = source.rule_id
        """
    )
    .whenMatchedUpdate(
        set={
            "column_name": "source.column_name",
            "rule_type": "source.rule_type",
            "rule_parameters": "source.rule_parameters",
            "severity": "source.severity",
            "failure_action": "source.failure_action",
            "execution_order": "source.execution_order",
            "is_active": "source.is_active",
            "rule_status": "source.rule_status",
            "description": "source.description",
            "updated_at": "source.updated_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("DQ rule configuration inserted / updated successfully.")

DQ rule configuration inserted / updated successfully.


In [0]:
# ------------------------------------------------------------
# 9. VERIFY REGISTERED DQ RULES
# ------------------------------------------------------------

registered_dq_df = (
    spark.read
    .format("delta")
    .load(DQ_CONFIG_PATH)
    .filter(F.col("dataset_name") == dataset_name)
)

display(
    registered_dq_df.orderBy("execution_order")
)

actual_count = registered_dq_df.count()

print(f"Total registered DQ rules: {actual_count}")

dataset_name,rule_id,column_name,rule_type,rule_parameters,severity,failure_action,execution_order,is_active,rule_status,description,created_at,updated_at
google_ads_campaigns,CHK_CUSTOMER_ID_NOT_NULL,customer_id,NOT_NULL,Map(),HIGH,QUARANTINE,1,true,APPROVED,Customer ID must not be null.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_CAMPAIGN_ID_NOT_NULL,campaign_id,NOT_NULL,Map(),HIGH,QUARANTINE,2,true,APPROVED,Campaign ID must not be null.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_CAMPAIGN_RESOURCE_NOT_NULL,campaign_resource_name,NOT_NULL,Map(),HIGH,QUARANTINE,3,true,APPROVED,Campaign resource name must not be null.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_SOURCE_EXTRACT_TS_NOT_NULL,source_extract_timestamp,NOT_NULL,Map(),HIGH,QUARANTINE,4,true,APPROVED,Source extraction timestamp must not be null.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_CAMPAIGN_RESOURCE_UNIQUE,campaign_resource_name,UNIQUE,Map(),HIGH,QUARANTINE,5,true,APPROVED,Campaign resource name must be unique.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_CAMPAIGN_STATUS_VALUES,campaign_status,ACCEPTED_VALUES,"Map(accepted_values -> ENABLED,PAUSED,REMOVED)",MEDIUM,QUARANTINE,6,true,APPROVED,Campaign status must belong to the approved values.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_ADVERTISING_CHANNEL_VALUES,advertising_channel_type,ACCEPTED_VALUES,"Map(accepted_values -> SEARCH,DISPLAY,VIDEO,SHOPPING,PERFORMANCE_MAX)",MEDIUM,QUARANTINE,7,true,APPROVED,Advertising channel type must be an approved value.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_CAMPAIGN_PRIMARY_STATUS_VALUES,campaign_primary_status,ACCEPTED_VALUES,"Map(accepted_values -> PAUSED,LIMITED,ELIGIBLE)",MEDIUM,QUARANTINE,8,true,APPROVED,Campaign primary status must be an approved value.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_SERVING_STATUS_VALUES,serving_status,ACCEPTED_VALUES,"Map(accepted_values -> NOT_SERVING,UNKNOWN,SERVING)",MEDIUM,QUARANTINE,9,true,APPROVED,Serving status must be an approved value.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z
google_ads_campaigns,CHK_TARGET_COUNTRY_FORMAT,target_country,REGEX,Map(pattern -> ^[A-Z]{2}$),MEDIUM,QUARANTINE,10,true,APPROVED,Target country must follow a two-letter uppercase code format.,2026-10-31T14:41:03.797Z,2026-11-15T01:59:46.102Z


Total registered DQ rules: 11
